# 04 - Preparar dados para modelagem

Este notebook carrega os Parquets intermediarios filtrados, seleciona descricoes de produtos, remove nulos e vazios, normaliza texto, consolida duplicatas preservando a frequencia de ocorrencia e salva a base final em `data/processed`.

In [1]:
from __future__ import annotations

import sys
import re
import unicodedata
import os
from pathlib import Path

import pandas as pd

for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    source_dir = path / "src"
    if source_dir.exists():
        sys.path.append(str(source_dir))
        break
else:
    raise FileNotFoundError("Nao encontrei a pasta src nos diretorios pais.")

from nofis_classifier.paths import find_project_root

## Configuracao

In [2]:
DATASET_NAME = "items-notas-fiscais-filtros-combinados"
FREQUENCY_COLUMN = "frequencia"

PROJECT_ROOT = find_project_root()
INTERIM_DIR = PROJECT_ROOT / "data" / "interim" / DATASET_NAME
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / DATASET_NAME

INTERIM_DIR, PROCESSED_DIR

(WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/interim/items-notas-fiscais-filtros-combinados'),
 WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/processed/items-notas-fiscais-filtros-combinados'))

## Funcoes de preparacao

In [3]:
def normalize_column_name(column_name: str) -> str:
    """Normalize a column name enough to detect expected fields."""
    text = unicodedata.normalize("NFKD", str(column_name))
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = text.upper()
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def find_column(columns: pd.Index, required_tokens: tuple[str, ...]) -> str:
    """Find the first column whose normalized name contains all required tokens."""
    for column in columns:
        normalized = normalize_column_name(column)
        if all(token in normalized for token in required_tokens):
            return column
    raise KeyError(f"Nao encontrei coluna com tokens {required_tokens}. Colunas: {list(columns)}")


def normalize_description(value: object) -> str:
    """Normalize product descriptions for TF-IDF and duplicate consolidation."""
    if pd.isna(value):
        return ""

    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def compact_month(period: str) -> str:
    """Convert YYYY-MM to YYYYMM for artifact names."""
    return period.replace("-", "")


def most_frequent_by_description(
    frame: pd.DataFrame,
    value_column: str,
    output_column: str,
) -> pd.DataFrame:
    """Return the most common value per normalized description."""
    valid = frame[["descricao_normalizada", value_column]].dropna(subset=[value_column]).copy()
    valid = valid[valid[value_column].astype(str).str.strip() != ""]
    if valid.empty:
        return pd.DataFrame(columns=["descricao_normalizada", output_column])

    weighted = (
        valid.groupby(["descricao_normalizada", value_column], as_index=False)
        .size()
        .sort_values(["descricao_normalizada", "size"], ascending=[True, False])
    )
    return weighted.drop_duplicates("descricao_normalizada")[["descricao_normalizada", value_column]].rename(
        columns={value_column: output_column}
    )

## Testes rapidos da normalizacao

In [4]:
normalization_examples = {
    " ARROZ TIPO 1 5KG ": "arroz tipo 1 5kg",
    "CAFE TORRADO 500g": "cafe torrado 500g",
    "LEITE UHT 1L": "leite uht 1l",
    "ACUCAR-CRISTAL 2 KG": "acucar cristal 2 kg",
}

for raw, expected in normalization_examples.items():
    normalized = normalize_description(raw)
    assert normalized == expected, (raw, normalized, expected)
    assert any(char.isalpha() for char in normalized), (raw, normalized)

informative_examples = ["QUEIJO MUCARELA, KG", "Couve Flor (Hortifruti) Ceasa", "PIMENTAO VERMELHA CEASA"]
for raw in informative_examples:
    normalized = normalize_description(raw)
    assert len(normalized) >= 3, (raw, normalized)
    assert any(char.isalpha() for char in normalized), (raw, normalized)

duplicate_check = pd.DataFrame(
    {
        "descricao_original": ["Arroz Tipo 1", " arroz tipo 1 ", "ARROZ TIPO 1"],
    }
)
duplicate_check["descricao_normalizada"] = duplicate_check["descricao_original"].map(normalize_description)
duplicate_frequency = duplicate_check.groupby("descricao_normalizada").size().iloc[0]
assert duplicate_frequency == len(duplicate_check)

## Carregar Parquets intermediarios

In [5]:
parquet_files = sorted(INTERIM_DIR.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"Nenhum Parquet encontrado em {INTERIM_DIR}")

frames = []
for parquet_file in parquet_files:
    frame = pd.read_parquet(parquet_file)
    frame["competencia"] = parquet_file.stem
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)

print(f"Arquivos carregados: {len(parquet_files)}")
print(f"Linhas intermediarias: {len(df):,}")
df.head()

Arquivos carregados: 4
Linhas intermediarias: 134,625


,CHAVE DE ACESSO,MODELO,SÉRIE,NÚMERO,NATUREZA DA OPERAÇÃO,DATA EMISSÃO,EVENTO MAIS RECENTE,DATA/HORA EVENTO MAIS RECENTE,CPF/CNPJ Emitente,RAZÃO SOCIAL EMITENTE,...,NÚMERO PRODUTO,DESCRIÇÃO DO PRODUTO/SERVIÇO,CÓDIGO NCM/SH,NCM/SH (TIPO DE PRODUTO),CFOP,QUANTIDADE,UNIDADE,VALOR UNITÁRIO,VALOR TOTAL,competencia
0,50250128991328000136550010000004261001326017,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,426,5405-Venda de mercadoria adquirida ou recebida...,01/01/2025 13:26:02,NaN,NaN,28991328000136,LOG COMERCIO DE PRODUTOS ALIMENTICIOS LTDA,...,1.0,"QUEIJO MUCARELA, KG",4061010.0,"Queijo tipo mussarela, fresco (não curado)",5405.0,"35,00",UNIDAD,"35,00","1225,00",2025-01
1,50250128991328000136550010000004271001332081,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,427,5405-Venda de mercadoria adquirida ou recebida...,01/01/2025 13:32:08,NaN,NaN,28991328000136,LOG COMERCIO DE PRODUTOS ALIMENTICIOS LTDA,...,1.0,Couve Flor (Hortifruti) Ceasa,7089000.0,"Outros legumes de vagem, frescos ou refrigerados",5405.0,"15,00",KG,"10,90","163,50",2025-01
2,50250128991328000136550010000004271001332081,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,427,5405-Venda de mercadoria adquirida ou recebida...,01/01/2025 13:32:08,NaN,NaN,28991328000136,LOG COMERCIO DE PRODUTOS ALIMENTICIOS LTDA,...,2.0,PIMENTAO VERMELHA CEASA,7096000.0,Pimentões e pimentas dos gêneros Capsicum ou P...,5405.0,"10,00",KG,"12,85","128,50",2025-01
3,50250128991328000136550010000004271001332081,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,427,5405-Venda de mercadoria adquirida ou recebida...,01/01/2025 13:32:08,NaN,NaN,28991328000136,LOG COMERCIO DE PRODUTOS ALIMENTICIOS LTDA,...,3.0,Pimentao Amarelo Ceasa,7096000.0,Pimentões e pimentas dos gêneros Capsicum ou P...,5405.0,"10,00",KG,"12,90","129,00",2025-01
4,50250128991328000136550010000004271001332081,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,427,5405-Venda de mercadoria adquirida ou recebida...,01/01/2025 13:32:08,NaN,NaN,28991328000136,LOG COMERCIO DE PRODUTOS ALIMENTICIOS LTDA,...,4.0,Abacaxi Cayena Ceasa,8043000.0,Abacaxis frescos ou secos,5405.0,"14,00",KG,"4,51","63,14",2025-01


## Selecionar descricoes

In [6]:
description_col = find_column(df.columns, ("DESCRI", "PRODUTO"))

support_columns = {
    "codigo_ncm": None,
    "ncm_tipo": None,
}

try:
    support_columns["codigo_ncm"] = find_column(df.columns, ("NCM", "SH"))
except KeyError:
    pass

try:
    support_columns["ncm_tipo"] = find_column(df.columns, ("TIPO", "PRODUTO"))
except KeyError:
    pass

selected_columns = [description_col, "competencia"]
selected_columns.extend(column for column in support_columns.values() if column is not None)

rename_columns = {description_col: "descricao_original"}
if support_columns["codigo_ncm"] is not None:
    rename_columns[support_columns["codigo_ncm"]] = "codigo_ncm"
if support_columns["ncm_tipo"] is not None:
    rename_columns[support_columns["ncm_tipo"]] = "ncm_tipo"

df_selected = df[selected_columns].rename(columns=rename_columns)

df_selected.head()

,descricao_original,competencia,codigo_ncm,ncm_tipo
0,"QUEIJO MUCARELA, KG",2025-01,4061010.0,"Queijo tipo mussarela, fresco (não curado)"
1,Couve Flor (Hortifruti) Ceasa,2025-01,7089000.0,"Outros legumes de vagem, frescos ou refrigerados"
2,PIMENTAO VERMELHA CEASA,2025-01,7096000.0,Pimentões e pimentas dos gêneros Capsicum ou P...
3,Pimentao Amarelo Ceasa,2025-01,7096000.0,Pimentões e pimentas dos gêneros Capsicum ou P...
4,Abacaxi Cayena Ceasa,2025-01,8043000.0,Abacaxis frescos ou secos


## Remover nulos, normalizar e consolidar duplicatas

In [7]:
input_rows = len(df_selected)

df_valid = df_selected.dropna(subset=["descricao_original"]).copy()
df_valid["descricao_original"] = df_valid["descricao_original"].astype(str).str.strip()
df_valid = df_valid[df_valid["descricao_original"] != ""].copy()

df_valid["descricao_normalizada"] = df_valid["descricao_original"].map(normalize_description)
df_valid = df_valid[df_valid["descricao_normalizada"] != ""].copy()

valid_rows = len(df_valid)

print(f"Linhas com coluna de descricao: {input_rows:,}")
print(f"Linhas com descricao valida: {valid_rows:,}")
print(f"Linhas removidas por nulos/vazios: {input_rows - valid_rows:,}")

Linhas com coluna de descricao: 134,625
Linhas com descricao valida: 134,624
Linhas removidas por nulos/vazios: 1


In [8]:
aggregations = {
    FREQUENCY_COLUMN: ("descricao_original", "size"),
    "descricao_original_exemplo": ("descricao_original", "first"),
    "descricoes_originais_distintas": ("descricao_original", "nunique"),
    "primeiro_mes": ("competencia", "min"),
    "ultimo_mes": ("competencia", "max"),
    "meses_presentes": ("competencia", "nunique"),
}

df_modeling = (
    df_valid.groupby("descricao_normalizada", as_index=False)
    .agg(**aggregations)
    .sort_values([FREQUENCY_COLUMN, "descricao_normalizada"], ascending=[False, True])
    .reset_index(drop=True)
)

if "codigo_ncm" in df_valid.columns:
    ncm_mode = most_frequent_by_description(df_valid, "codigo_ncm", "codigo_ncm_mais_frequente")
    df_modeling = df_modeling.merge(ncm_mode, on="descricao_normalizada", how="left")

if "ncm_tipo" in df_valid.columns:
    ncm_tipo_mode = most_frequent_by_description(df_valid, "ncm_tipo", "ncm_tipo_mais_frequente")
    df_modeling = df_modeling.merge(ncm_tipo_mode, on="descricao_normalizada", how="left")

df_modeling.head(10)

,descricao_normalizada,frequencia,descricao_original_exemplo,descricoes_originais_distintas,primeiro_mes,ultimo_mes,meses_presentes,codigo_ncm_mais_frequente,ncm_tipo_mais_frequente
0,cenoura,1533,CENOURA,2,2025-01,2025-04,4,7061000.0,"Cenouras e nabos, frescos ou refrigerados"
1,beterraba,1397,BETERRABA,2,2025-01,2025-04,4,7069000.0,"Beterrabas, rabanetes e outras raízes, frescas..."
2,melancia,1282,MELANCIA,3,2025-01,2025-04,4,8071100.0,Melancias frescas
3,batata inglesa,1022,BATATA INGLESA,3,2025-01,2025-04,4,7019000.0,"Batatas, frescas ou refrigeradas, exceto para ..."
4,batata doce,967,BATATA DOCE,5,2025-01,2025-04,4,7142000.0,"Batatas-doces, frescas, refrigeradas, congelad..."
5,banana prata,962,BANANA PRATA,4,2025-01,2025-04,4,8039000.0,"Bananas frescas ou secas, exceto bananas-da-terra"
6,pimentao verde,869,PIMENTAO VERDE,7,2025-01,2025-04,4,7096000.0,Pimentões e pimentas dos gêneros Capsicum ou P...
7,alface crespa,761,ALFACE CRESPA,3,2025-01,2025-04,4,7051900.0,Outras alfaces frescas ou refrigeradas
8,pepino,665,PEPINO,3,2025-01,2025-04,4,7070000.0,"Pepinos e pepininhos (cornichons), frescos ou ..."
9,cebola,648,CEBOLA,2,2025-01,2025-04,4,7031019.0,"Cebolas, frescas ou refrigeradas, exceto para ..."


## Validacoes finais

In [9]:
assert not df_modeling.empty
assert df_modeling["descricao_normalizada"].notna().all()
assert (df_modeling["descricao_normalizada"].str.strip() != "").all()
assert int(df_modeling[FREQUENCY_COLUMN].sum()) == valid_rows
assert df_modeling["descricao_normalizada"].is_unique

print(f"Descricoes unicas normalizadas: {len(df_modeling):,}")
print(f"Linhas validas consolidadas: {int(df_modeling[FREQUENCY_COLUMN].sum()):,}")
print(f"Frequencia preservada: {int(df_modeling[FREQUENCY_COLUMN].sum()) == valid_rows}")

Descricoes unicas normalizadas: 27,800
Linhas validas consolidadas: 134,624
Frequencia preservada: True


## Salvar base processada

In [11]:
start_period = compact_month(min(file.stem for file in parquet_files))
end_period = compact_month(max(file.stem for file in parquet_files))
output_path = PROCESSED_DIR / f"{DATASET_NAME}-{start_period}-{end_period}.parquet"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
if output_path.exists():
    raise FileExistsError(
        f"Arquivo processado ja existe: {output_path}. "
        "Escolha outro nome ou remova o arquivo manualmente se quiser regenerar."
    )

df_modeling.to_parquet(output_path, index=False)

output_path

WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/processed/items-notas-fiscais-filtros-combinados/items-notas-fiscais-filtros-combinados-202501-202504.parquet')

In [15]:
df_modeling.head()

,descricao_normalizada,frequencia,descricao_original_exemplo,descricoes_originais_distintas,primeiro_mes,ultimo_mes,meses_presentes,codigo_ncm_mais_frequente,ncm_tipo_mais_frequente
0,cenoura,1533,CENOURA,2,2025-01,2025-04,4,7061000.0,"Cenouras e nabos, frescos ou refrigerados"
1,beterraba,1397,BETERRABA,2,2025-01,2025-04,4,7069000.0,"Beterrabas, rabanetes e outras raízes, frescas..."
2,melancia,1282,MELANCIA,3,2025-01,2025-04,4,8071100.0,Melancias frescas
3,batata inglesa,1022,BATATA INGLESA,3,2025-01,2025-04,4,7019000.0,"Batatas, frescas ou refrigeradas, exceto para ..."
4,batata doce,967,BATATA DOCE,5,2025-01,2025-04,4,7142000.0,"Batatas-doces, frescas, refrigeradas, congelad..."


## Analise breve do dataset final

In [12]:
analysis_summary = pd.Series(
    {
        "arquivos_origem": len(parquet_files),
        "linhas_originais": len(df),
        "linhas_com_descricao_valida": valid_rows,
        "descricoes_unicas_normalizadas": len(df_modeling),
        "frequencia_total_preservada": int(df_modeling[FREQUENCY_COLUMN].sum()),
        "frequencia_media_por_descricao": df_modeling[FREQUENCY_COLUMN].mean(),
        "frequencia_mediana_por_descricao": df_modeling[FREQUENCY_COLUMN].median(),
        "maior_frequencia": int(df_modeling[FREQUENCY_COLUMN].max()),
        "periodo_inicial": df_modeling["primeiro_mes"].min(),
        "periodo_final": df_modeling["ultimo_mes"].max(),
    }
)

analysis_summary

arquivos_origem                           4
linhas_originais                     134625
linhas_com_descricao_valida          134624
descricoes_unicas_normalizadas        27800
frequencia_total_preservada          134624
frequencia_media_por_descricao      4.84259
frequencia_mediana_por_descricao        1.0
maior_frequencia                       1533
periodo_inicial                     2025-01
periodo_final                       2025-04
dtype: object

In [13]:
frequency_bins = pd.cut(
    df_modeling[FREQUENCY_COLUMN],
    bins=[0, 1, 2, 5, 10, 50, 100, float("inf")],
    labels=["1", "2", "3-5", "6-10", "11-50", "51-100", "101+"],
)

frequency_profile = (
    frequency_bins.value_counts(sort=False)
    .rename_axis("faixa_frequencia")
    .reset_index(name="descricoes")
)
frequency_profile["percentual"] = (
    frequency_profile["descricoes"] / len(df_modeling) * 100
).round(2)

frequency_profile

,faixa_frequencia,descricoes,percentual
0,1,15835,56.96
1,2,4455,16.03
2,3-5,4014,14.44
3,6-10,1672,6.01
4,11-50,1526,5.49
5,51-100,164,0.59
6,101+,134,0.48


In [14]:
top_descriptions = df_modeling[
    ["descricao_normalizada", "descricao_original_exemplo", FREQUENCY_COLUMN, "meses_presentes"]
].head(20)

top_descriptions

,descricao_normalizada,descricao_original_exemplo,frequencia,meses_presentes
0,cenoura,CENOURA,1533,4
1,beterraba,BETERRABA,1397,4
2,melancia,MELANCIA,1282,4
3,batata inglesa,BATATA INGLESA,1022,4
4,batata doce,BATATA DOCE,967,4
5,banana prata,BANANA PRATA,962,4
6,pimentao verde,PIMENTAO VERDE,869,4
7,alface crespa,ALFACE CRESPA,761,4
8,pepino,PEPINO,665,4
9,cebola,CEBOLA,648,4
